# Logit Model on Startup Success Data

This notebook builds the econometrics-ready dataset `econ_df` and estimates a logit model for startup success.

Model choices:
- dependent variable: `success` (`1=Success`, `0=Failed`)
- regressors: the final feature set from the EDA workflow, with `d_2_founders` (=1 if exactly two cofounders) for the core hypothesis on team size
- estimator: logistic regression via `statsmodels`
- covariance estimator: default maximum-likelihood standard errors


In [1]:
import warnings
import subprocess
import sys

warnings.filterwarnings('ignore')

package_specs = [
    ('statsmodels', 'statsmodels'),
    ('scikit-learn', 'sklearn'),
]
for pip_name, import_name in package_specs:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '--quiet'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid', palette='muted')
FIG_DPI = 140

df_raw = pd.read_csv('data/data_clean.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()


Raw shape: (472, 20)


,Dependent-Company Status,Number of Co-founders,B2C or B2B venture?,Time to 1st investment (in months),Number of of advisors,Team size Senior leadership,Employees count MoM change,Internet Activity Score,Last round of funding received (in milionUSD),Number of Investors in Angel and or VC,Number of of repeat investors,Presence of a top angel or venture fund in previous round of investment,Average Years of experience for founder and co founder,Have been part of successful startups in the past?,Degree from a Tier 1 or Tier 2 university?,Linear or Non-linear business model,"Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive",year of founding,Industry trend in investing,Disruptiveness of technology
0,Success,1,B2C,No Info,2,2,0.0,-1.0,0.45,0,4,Yes,High,No,Tier_1,Linear,Yes,No Info,2.0,Low
1,Success,2,B2C,10,0,4,NaN,125.0,5,0,0,No,High,Yes,Tier_1,Non-Linear,No,2011,3.0,Medium
2,Success,3,B2B,2,0,7,0.0,455.0,2.35,0,0,No,Medium,No,Tier_2,Non-Linear,No,2011,3.0,Medium
3,Success,2,B2C,1,0,4,10.0,-99.0,10.25,0,0,Yes,Medium,Yes,Tier_2,Non-Linear,No,2009,4.0,Medium
4,Success,1,B2B,13,1,8,3.0,496.0,5.5,0,0,No,High,No,NaN,Non-Linear,Yes,2010,3.0,Medium


## Build `econ_df`

This is the same econometrics-ready dataset used in the EDA: binary variables are encoded, the excluded variables are dropped, and the remaining gaps are handled with targeted imputation.


In [2]:
df = df_raw.copy()
df.replace('No Info', np.nan, inplace=True)

NUMERIC_COLS = [
    'Number of Co-founders',
    'Time to 1st investment (in months)',
    'Number of of advisors',
    'Team size Senior leadership',
    'Employees count MoM change',
    'Internet Activity Score',
    'Last round of funding received (in milionUSD)',
    'Number of Investors in Angel and or VC',
    'Number of of repeat investors',
    'year of founding',
    'Industry trend in investing',
]
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

CAT_COLS = [
    'Dependent-Company Status',
    'B2C or B2B venture?',
    'Presence of a top angel or venture fund in previous round of investment',
    'Average Years of experience for founder and co founder',
    'Have been part of successful startups in the past?',
    'Degree from a Tier 1 or Tier 2 university?',
    'Linear or Non-linear business model',
    'Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive',
    'Disruptiveness of technology',
]
for col in CAT_COLS:
    df[col] = df[col].astype(str).str.strip().replace('nan', np.nan)

econ_df = df.copy()

binary_dummy_specs = {
    'Dependent-Company Status': ('success', {'Success': 1, 'Failed': 0}),
    'Presence of a top angel or venture fund in previous round of investment': ('d_top_fund', {'Yes': 1, 'No': 0}),
    'Have been part of successful startups in the past?': ('d_prior_success', {'Yes': 1, 'No': 0}),
    'Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive': ('d_capital_intensive', {'Yes': 1, 'No': 0}),
    'B2C or B2B venture?': ('d_b2b', {'B2B': 1, 'B2C': 0}),
    'Linear or Non-linear business model': ('d_nonlinear', {'Non-Linear': 1, 'Linear': 0}),
}
for source_col, (dummy_col, mapping) in binary_dummy_specs.items():
    econ_df[dummy_col] = econ_df[source_col].map(mapping).astype('Int64')

ordinal_specs = {
    'Average Years of experience for founder and co founder': ('founder_experience_score', {'Low': 1, 'Medium': 2, 'High': 3}),
    'Disruptiveness of technology': ('tech_disruptiveness_score', {'Low': 1, 'Medium': 2, 'High': 3}),
}
for source_col, (score_col, mapping) in ordinal_specs.items():
    econ_df[score_col] = econ_df[source_col].map(mapping).astype('Int64')

econ_df = econ_df.drop(columns=[
    'Dependent-Company Status',
    'Presence of a top angel or venture fund in previous round of investment',
    'Have been part of successful startups in the past?',
    'Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive',
    'B2C or B2B venture?',
    'Linear or Non-linear business model',
    'Average Years of experience for founder and co founder',
    'Degree from a Tier 1 or Tier 2 university?',
    'Disruptiveness of technology',
    'Last round of funding received (in milionUSD)',
    'Employees count MoM change',
])

ANALYSIS_FEATURES = [
    'd_2_founders',
    'Time to 1st investment (in months)',
    'Number of of advisors',
    'Team size Senior leadership',
    'Internet Activity Score',
    'Number of Investors in Angel and or VC',
    'Number of of repeat investors',
    'year of founding',
    'Industry trend in investing',
    'd_top_fund',
    'd_prior_success',
    'd_capital_intensive',
    'd_b2b',
    'd_nonlinear',
    'founder_experience_score',
    'tech_disruptiveness_score',
]

def first_mode(series):
    non_null = series.dropna()
    if non_null.empty:
        return np.nan
    return non_null.mode().iloc[0]

def targeted_fill(frame, column, group_sets, strategy='median', round_result=False, cast_to_int=False):
    filled = frame[column].copy()
    before = int(filled.isna().sum())
    if before == 0:
        if cast_to_int:
            filled = filled.astype('Int64')
        return filled, before, before

    working = frame.copy()
    working[column] = filled

    for group_cols in group_sets:
        if strategy == 'median':
            stat = working.groupby(group_cols)[column].transform('median')
        else:
            stat = working.groupby(group_cols)[column].transform(first_mode)
        filled = filled.fillna(stat)
        working[column] = filled

    if strategy == 'median':
        fallback = filled.median()
    else:
        fallback_mode = filled.dropna().mode()
        fallback = fallback_mode.iloc[0] if not fallback_mode.empty else np.nan

    filled = filled.fillna(fallback)
    if round_result:
        filled = filled.round()
    if cast_to_int:
        filled = filled.astype('Int64')

    after = int(filled.isna().sum())
    return filled, before, after

imputation_plan = [
    ('d_b2b', [['d_nonlinear'], ['d_capital_intensive']], 'mode', True, True),
    ('d_nonlinear', [['d_b2b'], ['d_capital_intensive']], 'mode', True, True),
    ('d_capital_intensive', [['d_b2b', 'd_nonlinear'], ['d_b2b']], 'mode', True, True),
    ('d_top_fund', [['d_b2b', 'd_prior_success'], ['d_b2b']], 'mode', True, True),
    ('d_prior_success', [['d_top_fund', 'd_b2b'], ['d_top_fund']], 'mode', True, True),
    ('year of founding', [['d_b2b', 'd_nonlinear'], ['d_b2b']], 'median', True, True),
    ('Industry trend in investing', [['year of founding', 'd_b2b'], ['year of founding']], 'median', True, True),
    ('founder_experience_score', [['d_prior_success', 'd_top_fund'], ['d_prior_success']], 'median', True, True),
    ('tech_disruptiveness_score', [['d_nonlinear', 'd_capital_intensive'], ['d_nonlinear']], 'median', True, True),
    ('Time to 1st investment (in months)', [['d_top_fund', 'd_prior_success', 'd_b2b'], ['d_top_fund', 'd_b2b'], ['d_b2b']], 'median', True, True),
    ('Internet Activity Score', [['d_b2b', 'year of founding'], ['d_b2b']], 'median', False, False),
    ('Number of Investors in Angel and or VC', [['d_top_fund', 'd_prior_success'], ['d_top_fund']], 'median', True, True),
    ('Number of of repeat investors', [['d_top_fund', 'd_prior_success'], ['d_top_fund']], 'median', True, True),
]

imputation_log = []
for column, group_sets, strategy, round_result, cast_to_int in imputation_plan:
    econ_df[column], missing_before, missing_after = targeted_fill(
        econ_df, column, group_sets, strategy=strategy,
        round_result=round_result, cast_to_int=cast_to_int,
    )
    imputation_log.append({
        'feature': column,
        'strategy': strategy,
        'missing_before': missing_before,
        'missing_after': missing_after,
    })

imputation_summary = pd.DataFrame(imputation_log)

# D_{2 founders}: 1 iff exactly two cofounders (hypothesis tests use this, not a linear count)
if econ_df['Number of Co-founders'].isna().any():
    econ_df['Number of Co-founders'] = econ_df['Number of Co-founders'].fillna(
        econ_df['Number of Co-founders'].median()
    )
econ_df['d_2_founders'] = (econ_df['Number of Co-founders'].astype(float) == 2).astype(float)

X = econ_df[ANALYSIS_FEATURES].astype(float)
y = econ_df['success'].astype(float)

print(f'econ_df shape: {econ_df.shape}')
print(f'Total missing cells in X: {int(X.isna().sum().sum())}')
display(imputation_summary[imputation_summary['missing_before'] > 0])
display(X.head())


econ_df shape: (472, 18)
Total missing cells in X: 0


,feature,strategy,missing_before,missing_after
0,d_b2b,mode,3,0
1,d_nonlinear,mode,18,0
2,d_capital_intensive,mode,26,0
3,d_top_fund,mode,97,0
4,d_prior_success,mode,20,0
5,year of founding,median,59,0
6,Industry trend in investing,median,82,0
7,founder_experience_score,median,80,0
8,tech_disruptiveness_score,median,82,0
9,Time to 1st investment (in months),median,96,0


,d_2_founders,Time to 1st investment (in months),Number of of advisors,Team size Senior leadership,Internet Activity Score,Number of Investors in Angel and or VC,Number of of repeat investors,year of founding,Industry trend in investing,d_top_fund,d_prior_success,d_capital_intensive,d_b2b,d_nonlinear,founder_experience_score,tech_disruptiveness_score
0,0.0,10.0,2.0,2.0,-1.0,0.0,4.0,2010.0,2.0,1.0,0.0,1.0,0.0,0.0,3.0,1.0
1,1.0,10.0,0.0,4.0,125.0,0.0,0.0,2011.0,3.0,0.0,1.0,0.0,0.0,1.0,3.0,2.0
2,0.0,2.0,0.0,7.0,455.0,0.0,0.0,2011.0,3.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0
3,1.0,1.0,0.0,4.0,-99.0,0.0,0.0,2009.0,4.0,1.0,1.0,0.0,0.0,1.0,2.0,2.0
4,0.0,13.0,1.0,8.0,496.0,0.0,0.0,2010.0,3.0,0.0,0.0,1.0,1.0,1.0,3.0,2.0


## Estimate Logit

Because the dependent variable is binary, the model is estimated as a logistic regression. To penalize over-fitting, the current specification is treated as a candidate feature pool and screened with a stepwise AIC search while always keeping `d_2_founders` in the model for the core team-size hypothesis. The notebook reports both the full candidate model and the AIC-selected model, then uses the selected specification for inference and diagnostics. Coefficients are reported in log-odds units, and the table below also includes odds ratios for easier interpretation. The final reported model uses the default maximum-likelihood standard errors.


In [ ]:
full_features = ANALYSIS_FEATURES.copy()
forced_features = ['d_2_founders']


def fit_logit(feature_list):
    X_curr = sm.add_constant(X[feature_list], has_constant='add')
    result = sm.Logit(y, X_curr).fit(disp=False)
    return result, X_curr


def evaluate_feature_set(feature_list):
    try:
        result, X_curr = fit_logit(feature_list)
    except Exception:
        return None

    return {
        'features': list(feature_list),
        'result': result,
        'X': X_curr,
        'aic': float(result.aic),
    }


def stepwise_aic_selection(candidate_features, forced_features, tol=1e-8):
    selected = list(dict.fromkeys(forced_features))
    current = evaluate_feature_set(selected)
    if current is None:
        raise ValueError('Could not fit the initial forced-feature model for AIC selection.')

    history = [{
        'step': 0,
        'action': 'start',
        'feature': ', '.join(selected),
        'n_features': len(selected),
        'aic': current['aic'],
        'delta_aic': np.nan,
    }]

    while True:
        candidate_moves = []

        for feature in candidate_features:
            if feature in selected:
                continue
            trial = evaluate_feature_set(selected + [feature])
            if trial is not None:
                candidate_moves.append({
                    'action': 'add',
                    'feature': feature,
                    **trial,
                })

        for feature in selected:
            if feature in forced_features:
                continue
            trial = evaluate_feature_set([col for col in selected if col != feature])
            if trial is not None:
                candidate_moves.append({
                    'action': 'drop',
                    'feature': feature,
                    **trial,
                })

        if not candidate_moves:
            break

        best_move = min(candidate_moves, key=lambda move: move['aic'])
        delta_aic = best_move['aic'] - current['aic']
        if delta_aic >= -tol:
            break

        selected = best_move['features']
        current = best_move
        history.append({
            'step': len(history),
            'action': best_move['action'],
            'feature': best_move['feature'],
            'n_features': len(selected),
            'aic': current['aic'],
            'delta_aic': delta_aic,
        })

    history_df = pd.DataFrame(history)
    return selected, current['result'], current['X'], history_df


full_logit_results, X_logit_full = fit_logit(full_features)
selected_features, logit_results, X_logit, aic_path = stepwise_aic_selection(
    full_features,
    forced_features,
)

model_comparison = pd.DataFrame([
    {
        'specification': 'Full candidate model',
        'n_features': len(full_features),
        'AIC': full_logit_results.aic,
        'BIC': full_logit_results.bic,
        'Pseudo_R2': full_logit_results.prsquared,
    },
    {
        'specification': 'AIC-selected model',
        'n_features': len(selected_features),
        'AIC': logit_results.aic,
        'BIC': logit_results.bic,
        'Pseudo_R2': logit_results.prsquared,
    },
]).sort_values('AIC')

selected_features_df = pd.DataFrame({'selected_feature': selected_features})

print('AIC-selected features')
display(selected_features_df)
print('Model comparison')
display(model_comparison.round(3))
print('AIC search path')
display(aic_path.round({'aic': 3, 'delta_aic': 3}))
print(logit_results.summary())

conf_int = logit_results.conf_int()
coef_table = pd.DataFrame({
    'coef_log_odds': logit_results.params,
    'std_error': logit_results.bse,
    'z': logit_results.tvalues,
    'p_value': logit_results.pvalues,
    'ci_low': conf_int[0],
    'ci_high': conf_int[1],
    'odds_ratio': np.exp(logit_results.params),
    'or_ci_low': np.exp(conf_int[0]),
    'or_ci_high': np.exp(conf_int[1]),
})
coef_table = coef_table.sort_values('p_value')
display(coef_table)

marginal_effects = logit_results.get_margeff(at='mean')
print(marginal_effects.summary())


### Core hypotheses (one-sided Wald tests)

Using the asymptotic normal Wald statistic $z=\hat\beta/\mathrm{SE}(\hat\beta)$ from the fitted logit:

| Hypothesis | $H_0$ | $H_1$ | one-sided $p$-value |
|------------|-------|-------|---------------------|
| Two cofounders | $\beta_{\texttt{d\_2\_founders}} \leq 0$ | $\beta > 0$ | $\Phi(-z)$ |
| B2B | $\beta_{\texttt{d\_b2b}} \leq 0$ | $\beta > 0$ | $\Phi(-z)$ |
| Time to first investment | $\beta_{\texttt{Time}} \geq 0$ | $\beta < 0$ | $\Phi(z)$ |

Reject $H_0$ at level $\alpha$ when $p<\alpha$ (default $\alpha=0.05$). Two-sided confidence intervals in `coef_table` remain the usual 95% intervals; one-sided inference uses the row below.

In [4]:
import math

ALPHA = 0.05


def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def p_greater_than_zero(coef, se):
    """H0: beta <= 0 vs H1: beta > 0 (right-tail)."""
    z = coef / se
    return float(1.0 - _norm_cdf(z))


def p_less_than_zero(coef, se):
    """H0: beta >= 0 vs H1: beta < 0 (left-tail)."""
    z = coef / se
    return float(_norm_cdf(z))


hypothesis_specs = [
    (
        'd_2_founders',
        r'H0: beta <= 0 vs H1: beta > 0',
        p_greater_than_zero,
    ),
    (
        'd_b2b',
        r'H0: beta <= 0 vs H1: beta > 0',
        p_greater_than_zero,
    ),
    (
        'Time to 1st investment (in months)',
        r'H0: beta >= 0 vs H1: beta < 0',
        p_less_than_zero,
    ),
]

rows = []
for param_name, h_text, p_fn in hypothesis_specs:
    if param_name not in logit_results.params.index:
        rows.append({
            'parameter': param_name,
            'selected_by_aic': False,
            'hypothesis': h_text,
            'coef': np.nan,
            'std_error': np.nan,
            'z': np.nan,
            'p_one_sided': np.nan,
            f'reject_H0_alpha_{int(ALPHA * 100)}': np.nan,
        })
        continue

    b = float(logit_results.params[param_name])
    se = float(logit_results.bse[param_name])
    z = b / se
    p1 = p_fn(b, se)
    rows.append({
        'parameter': param_name,
        'selected_by_aic': True,
        'hypothesis': h_text,
        'coef': b,
        'std_error': se,
        'z': z,
        'p_one_sided': p1,
        f'reject_H0_alpha_{int(ALPHA * 100)}': p1 < ALPHA,
    })

hypothesis_df = pd.DataFrame(rows)
display(hypothesis_df)

,parameter,selected_by_aic,hypothesis,coef,std_error,z,p_one_sided,reject_H0_alpha_5
0,d_2_founders,True,H0: beta <= 0 vs H1: beta > 0,0.326996,0.266995,1.224727,0.110339,False
1,d_b2b,True,H0: beta <= 0 vs H1: beta > 0,0.873955,0.279014,3.132300,0.000867,True
2,Time to 1st investment (in months),True,H0: beta >= 0 vs H1: beta < 0,0.046263,0.011767,3.931465,0.999958,False


## Diagnostics

This section reports fit measures, ROC-AUC discrimination, a simple classification check at the `0.5` threshold, calibration by predicted-probability deciles, and a multicollinearity screen via VIF for the AIC-selected logit specification.


In [ ]:
pred_prob = logit_results.predict(X_logit)
pred_class = (pred_prob >= 0.5).astype(int)
accuracy = (pred_class == y).mean()
roc_auc = roc_auc_score(y, pred_prob)
fpr, tpr, roc_thresholds = roc_curve(y, pred_prob)
confusion = pd.crosstab(
    pd.Series(y.astype(int), name='Actual'),
    pd.Series(pred_class.astype(int), name='Predicted'),
    dropna=False,
)

roc_points = pd.DataFrame({
    'false_positive_rate': fpr,
    'true_positive_rate': tpr,
    'threshold': roc_thresholds,
})
roc_preview = pd.concat([
    roc_points.head(5),
    roc_points.tail(5),
]).drop_duplicates().reset_index(drop=True)

vif = pd.DataFrame({
    'feature': X_logit.columns,
    'VIF': [variance_inflation_factor(X_logit.values, i) for i in range(X_logit.shape[1])],
}).sort_values('VIF', ascending=False)

print(f'Log-likelihood: {logit_results.llf:.4f}')
print(f'Pseudo R-squared (McFadden): {logit_results.prsquared:.4f}')
print(f'AIC: {logit_results.aic:.2f}')
print(f'BIC: {logit_results.bic:.2f}')
print(f'ROC-AUC: {roc_auc:.3f}')
print(f'Classification accuracy at 0.5 cutoff: {accuracy:.1%}')

print('\nConfusion matrix')
display(confusion)

print('ROC curve checkpoints')
display(roc_preview.round(3))

print('VIF table')
display(vif)

prob_df = pd.DataFrame({'pred_prob': pred_prob, 'success': y})
prob_df['prob_bin'] = pd.qcut(prob_df['pred_prob'], q=10, duplicates='drop')
calibration = prob_df.groupby('prob_bin').agg(
    mean_predicted=('pred_prob', 'mean'),
    actual_rate=('success', 'mean'),
    n=('success', 'size'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 4), dpi=FIG_DPI)
axes[0].hist(pred_prob, bins=25, color='#2ecc71', edgecolor='white')
axes[0].set_title('Distribution of predicted probabilities')
axes[0].set_xlabel('Predicted success probability')
axes[0].set_ylabel('Count')

axes[1].plot(fpr, tpr, color='#e67e22', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1, label='Random classifier')
axes[1].set_title('ROC curve')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].legend(loc='lower right')

axes[2].plot(calibration['mean_predicted'], calibration['actual_rate'], 'o-', color='#3498db')
axes[2].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
axes[2].set_title('Calibration by probability decile')
axes[2].set_xlabel('Mean predicted probability')
axes[2].set_ylabel('Observed success rate')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()
